# 1. Decoding

**Decoding** is the process of converting a language model's output logits into a sequence of tokens, one token at a time, until a stopping condition is reached.

The model predicts the next token based on the input sequence and the tokens generated so far. Decoding determines how the next token is selected from those predictions.

## 1.1 Decoding Pipeline

A typical autoregressive decoding process follows these steps:

1. **Input tokens:** Tokenize the input text and pass the resulting token IDs to the model.
2. **Forward pass:** The model produces logits for each position in the input sequence.
3. **Select the last logits:** Extract the logits corresponding to the final sequence position to predict the next token.
4. **Apply temperature:** Adjust the logits to control the randomness of token selection.
5. **Apply sampling filters:** Optionally use strategies such as Top-K or Top-P to restrict the candidate tokens.
6. **Apply Softmax:** Convert the processed logits into probabilities.
7. **Select the next token:** Choose the next token using greedy decoding or a sampling strategy.
8. **Append the token:** Add the selected token to the input sequence.
9. **Repeat:** Run the process again until an end-of-sequence token or another stopping condition is reached.

The exact order of temperature, filtering, and Softmax depends on the decoding implementation. For example, Top-K can be applied to temperature-scaled logits before Softmax.

## 1.2 Common Decoding Strategies

- **Greedy Decoding:** Selects the token with the highest probability at every step.
- **Temperature Sampling:** Adjusts the probability distribution to control randomness.
- **Top-K Sampling:** Restricts selection to the \(K\) tokens with the highest scores.
- **Top-P (Nucleus) Sampling:** Selects from the smallest set of tokens whose cumulative probability reaches a specified threshold.

These strategies can be combined. For example, a model may use temperature scaling followed by Top-K filtering and probabilistic sampling.


## 1.3 Autoregressive Generation

In autoregressive generation, the model predicts **one new token at a time**, using the previous tokens as context.

Suppose the input is:

`The history of Brazil began`

The generation process follows these steps:

1. The model receives the input tokens.
2. It produces logits with shape \((B, T, V)\), where \(B\) is the batch size, \(T\) is the sequence length, and \(V\) is the vocabulary size.
3. The last position's logits are extracted, resulting in a tensor of shape \((B, V)\).
4. The decoding strategy selects the next token.
5. The selected token is appended to the input sequence.
6. The updated sequence is passed to the model again to predict the next token.

This process repeats until an end-of-sequence token is generated or a stopping condition is reached.

**Example:**

```text
Input:      The history of Brazil began
Token 1:    in
Token 2:    1500
Token 3:    when
Token 4:    Portuguese
...
```

Each generated token becomes part of the context for the next prediction.

Mathematically, the model generates a sequence according to:

$P(x_1,\ldots,x_T)=\prod_{t=1}^{T} P(x_t \mid x_1,\ldots,x_{t-1})$

Each token is conditioned on the preceding tokens, which is why this process is called *autoregressive generation*.

**Key takeaway:** The model predicts the next token from the existing context, the decoding strategy selects it, and the process repeats to generate the sequence.

# 1. Model Output

The model receives a sequence of input token IDs and produces **logits** for every position in the sequence and every token in the vocabulary.

For a causal language model, the output tensor typically has the shape:

\[
(B,T,V)
\]

Where:

- \(B\): Batch size.
- \(T\): Sequence length.
- \(V\): Vocabulary size.

For example, if the batch contains 1 sequence with 10 tokens and the vocabulary has 50,257 tokens, the output shape is:

\[
(1,10,50257)
\]

Each position contains a vector of logits representing the model's scores for predicting the next token.

During autoregressive decoding, we extract the logits at the **last sequence position** to predict the next token:

```python
logits = model(input_ids)

last_logits = logits[:, -1, :]
```

If `input_ids` has shape `(B, T)`, then `last_logits` has shape `(B, V)`.

These logits are then passed to a decoding strategy, such as Greedy Decoding, Sampling, or Top-K Sampling, to select the next token.

In [81]:
import torch

input_ids = torch.randint(1, 10, (1, 10))
logits = torch.rand(3, 5)
print(f'Original Logits:\n{input_ids}')
print(f'Original Logits matrixs:\n{logits}')

Original Logits:
tensor([[6, 3, 5, 9, 4, 5, 5, 4, 7, 8]])
Original Logits matrixs:
tensor([[0.1746, 0.5532, 0.8435, 0.5165, 0.8443],
        [0.1010, 0.2267, 0.2077, 0.0577, 0.3894],
        [0.2676, 0.5273, 0.6594, 0.5360, 0.8520]])


# 2. Extracting the Last Logits

During autoregressive decoding, we only need the logits from the **last sequence position** to predict the next token.

Given a model output with shape \((B,T,V)\), we can extract the last logits using:

```python
last_logits = logits[:, -1, :]
```

- `:` selects every sequence in the batch.
- `-1` selects the last position in the sequence.
- `:` selects all vocabulary logits at that position.

### Example

If the model output has shape:

```python
logits.shape  # (1, 10, 50257)
```

After extracting the last logits:

```python
last_logits = logits[:, -1, :]

last_logits.shape  # (1, 50257)
```

The resulting tensor contains one logit for each vocabulary token, representing the model's scores for the next-token prediction.

**Key takeaway:** We extract the last logits because they represent the predictions for the next token after the current input sequence.

---
# 3. Strategies I (Choose next token)

## 3.1 Greedy Decoding

In **Greedy Decoding**, the objective is to select the token with the highest logit at each generation step.

We can use `torch.argmax()` to find the index of the highest value in a tensor:

```python
arg_max = torch.argmax(tensor, dim=-1)
```

Basically, `torch.argmax()` returns the **index of the highest value** along the specified dimension.

For example, if the last logits are:

```python
logits = torch.tensor([[2.1, 5.7, 1.3, 3.4]])

next_token = torch.argmax(logits, dim=-1)

# tensor([1])
```

The selected token has index `1`, because `5.7` is the highest logit.

> **Remember:** Softmax is unnecessary for Greedy Decoding. Since Softmax preserves the ranking of logits, the highest logit will also have the highest probability. Applying Softmax would add an unnecessary computation if we only need to select the token with the highest score.

In [63]:
import torch.nn.functional as F

last_logit = logits[-1, :].unsqueeze(dim=0) # Get the last logit
next_token = torch.argmax(last_logit, keepdim = True) # Get higher prob of the last logit, will return a (1, 1)
new_sequence = torch.cat([input_ids, next_token], dim = 1)

print(f'Last logit values: {last_logit}, shape: {last_logit.shape}')
print(f'So the next token (IDX) choose will be: {next_token}')
print(f'So now the sequence updated will be:{new_sequence}, \nand the original its:{input_ids}')

Last logit values: tensor([[0.3145, 0.9261, 0.8961, 0.3079, 0.0105]]), shape: torch.Size([1, 5])
So the next token (IDX) choose will be: tensor([[1]])
So now the sequence updated will be:tensor([[1, 4, 5, 6, 8, 1, 7, 9, 5, 4, 1]]), 
and the original its:tensor([[1, 4, 5, 6, 8, 1, 7, 9, 5, 4]])


## 3.2 Sampling

Unlike Greedy Decoding, **Sampling** does not always select the token with the highest probability. Instead, it converts the logits into a probability distribution and randomly selects a token according to those probabilities.

For this, we use `F.softmax()` and `torch.multinomial()`.

```python
import torch
import torch.nn.functional as F

tensor_softmax = F.softmax(tensor, dim=-1)

next_token = torch.multinomial(
    tensor_softmax,
    num_samples=1
)
```

### How it works

1. **Softmax:** Converts the logits into probabilities that sum to 1 along the specified dimension.
2. **Multinomial:** Randomly samples a token index according to the probability distribution.

For example, suppose the model produces these probabilities:

| Token | Probability |
|---|---:|
| `tokenX` | 0.70 |
| `tokenY` | 0.20 |
| `tokenZ` | 0.10 |

Sampling gives `tokenX` a 70% chance of being selected, `tokenY` a 20% chance, and `tokenZ` a 10% chance.

Therefore, a token with a lower probability can still be selected.

**Important:** `torch.multinomial()` returns the **index of the sampled token**, not the token itself. The index can then be used to retrieve the corresponding token ID from the model's vocabulary.

Unlike Greedy Decoding, Sampling introduces randomness into generation, allowing different outputs even when the input and logits remain the same.



In [66]:
import torch.nn.functional as F

last_logit = logits[-1, :].unsqueeze(dim=0) # Get the last logit

last_logit_softmax = F.softmax(last_logit, dim = 1)

next_token = torch.multinomial(last_logit_softmax, num_samples = 1) 

new_sequence = torch.cat([input_ids, next_token], dim = 1)

print(f'Last logit values: {last_logit}, shape: {last_logit.shape}')
print(f'Last logit values after the softmax: {last_logit_softmax}')
print(f'So the next token (IDX) choose will be: {next_token}')
print(f'So now the sequence updated will be:{new_sequence}, \nand the original its:{input_ids}')

Last logit values: tensor([[0.3145, 0.9261, 0.8961, 0.3079, 0.0105]]), shape: torch.Size([1, 5])
Last logit values after the softmax: tensor([[0.1571, 0.2897, 0.2811, 0.1561, 0.1160]])
So the next token (IDX) choose will be: tensor([[3]])
So now the sequence updated will be:tensor([[1, 4, 5, 6, 8, 1, 7, 9, 5, 4, 3]]), 
and the original its:tensor([[1, 4, 5, 6, 8, 1, 7, 9, 5, 4]])


---
# 4. Strategies II (Better logits values)

## 4.1 Temperature

**Temperature** controls the randomness of token selection by adjusting the model's logits before applying Softmax.

- **Low temperature (\(T < 1\)):** Makes the probability distribution more concentrated, favoring tokens with higher probabilities.
- **High temperature (\(T > 1\)):** Makes the probability distribution flatter, increasing the relative chances of lower-probability tokens.
- **Temperature approaching infinity (\(T \to \infty\)):** Makes the distribution approach a uniform distribution, assuming finite logits and a finite vocabulary.

### 4.1.1 How Temperature Affects Token Probabilities

Consider a model choosing between two possible tokens, `tokenX` and `tokenY`.

| Temperature (\(T\)) | P(tokenX) | P(tokenY) |
|---:|---:|---:|
| 0.5 | 0.917 | 0.083 |
| 1.0 | 0.769 | 0.231 |
| 2.0 | 0.646 | 0.354 |
| 5.0 | 0.560 | 0.440 |
| \(T \to \infty\) | 0.500 | 0.500 |

As temperature increases, the probability distribution becomes more uniform. As temperature decreases, the model becomes more confident in its highest-probability choices.

### 4.1.2 Low Temperature: More Predictable Outputs

When \(T < 1\), the model favors tokens with higher probabilities, making its outputs more predictable and consistent.

Consider a training dataset in which a particular context is followed by:

- `tokenX`: 10 occurrences
- `tokenY`: 3 occurrences

If the model learns this pattern, it may assign a higher probability to `tokenX` than to `tokenY`.

With a low temperature, the probability of selecting `tokenX` becomes even higher relative to `tokenY`.

This can make the model favor common patterns and sequences learned during training. It may therefore produce outputs that resemble frequent training patterns more consistently, sometimes repeating familiar sequences.

**Important:** Low temperature does not make the model literally copy its training data. It changes how the model samples from the probability distribution it has learned.

### 4.1.3 High Temperature: More Varied Outputs

When \(T > 1\), the model produces a flatter probability distribution, giving lower-probability tokens a greater chance of being selected.

Using the same example:

- `tokenX`: 10 occurrences
- `tokenY`: 3 occurrences

At a higher temperature, `tokenY` becomes more likely relative to `tokenX`. However, their probabilities become equal only if the adjusted logits are equal or the temperature approaches infinity in the two-token example.

This introduces more variation into the generated text and allows the model to explore less common continuations.

However, higher temperature does not necessarily improve the model's ability to adapt or understand a context. It increases sampling diversity, potentially at the cost of coherence, factual accuracy, and consistency.

**Important:** High temperature does not calculate an average of the training data. It flattens the model's learned probability distribution, making less likely alternatives more accessible.

### 4.1.4 Recommended Temperature by Task

The following ranges are practical starting points, not strict rules. The ideal temperature depends on the model, the task, and the desired balance between predictability and diversity.

| Task | Recommended Temperature | Reason |
|---|---:|---|
| Text classification | 0.0–0.3 | Consistent outputs across predefined categories. |
| Structured data extraction (JSON) | 0.0–0.2 | Predictable, structured outputs. |
| Code generation | 0.1–0.5 | Favors conventional and coherent solutions. |
| Customer support assistants | 0.3–0.7 | Natural responses without unnecessary variation. |
| Factual question answering | 0.1–0.5 | Prioritizes consistency over creativity. |
| Educational explanations | 0.5–0.8 | Allows different ways of explaining concepts. |
| Casual conversations | 0.7–1.0 | Encourages variety in language and tone. |
| Creative writing and storytelling | 0.8–1.2 | Explores more diverse narrative continuations. |
| Brainstorming and idea generation | 1.0–1.5 | Encourages less obvious alternatives. |
| Experimental and surreal generation | 1.3–2.0+ | Accepts reduced coherence in exchange for unexpected outputs. |

*Note: A temperature of 0 is commonly used to request greedy decoding, where the highest-scoring token is selected at each step. Exact behavior depends on the implementation. Temperature alone does not guarantee deterministic outputs.*

### 4.1.5 Key Takeaway

- **Low temperature:** More predictable outputs, stronger preference for high-probability tokens, and less variation.
- **Temperature = 1:** Preserves the original Softmax probability distribution.
- **High temperature:** More diverse outputs, greater relative probability for less likely tokens, and potentially less coherence.
- **Temperature approaching infinity:** Approaches a uniform distribution over the vocabulary.

Temperature controls **how the model selects among its predicted tokens**, not what the model learned during training.

In [79]:
last_logit = logits[-1, :].unsqueeze(dim=0) 

temperature = 1.5
last_logit_scaled = last_logit / temperature

last_logit_softmax = F.softmax(last_logit, dim = 1)
last_logit_scaled_softmax = F.softmax(last_logit_scaled, dim = 1)

# Now you decide if greedy or sampling (MOST COMMON IN SAMPLING)
next_token_scaled = torch.multinomial(last_logit_scaled_softmax, num_samples=1)


print(f'Last logit values: {last_logit}')
print(f'Last logit values now scaled: {last_logit_scaled}')

print(f'=' * 50)
print(f'Difference of NORMAL X SCALED WITH SOFTMAX')
print(f'Normal: {last_logit_softmax}')
print(f'Scaled: {last_logit_scaled_softmax}')




Last logit values: tensor([[0.3145, 0.9261, 0.8961, 0.3079, 0.0105]])
Last logit values now scaled: tensor([[0.2097, 0.6174, 0.5974, 0.2053, 0.0070]])
Difference of NORMAL X SCALED WITH SOFTMAX
Normal: tensor([[0.1571, 0.2897, 0.2811, 0.1561, 0.1160]])
Scaled: tensor([[0.1727, 0.2597, 0.2546, 0.1720, 0.1410]])


## 4.2 Top-K Sampling

**Top-K Sampling** restricts the candidate tokens to the \(K\) tokens with the highest logits. Tokens outside this set are excluded from sampling.

We can use `torch.topk()` to retrieve the \(K\) highest values and their indices.

```python
topk_values, topk_indices = torch.topk(
    logits,
    k=N,
    dim=-1
)
```

- `topk_values`: The \(K\) highest logits.
- `topk_indices`: The indices of the corresponding tokens.
- `k=N`: The number of candidate tokens to keep.

**Important:** `torch.topk()` does not directly perform Top-K Sampling. It only retrieves the highest values and their indices. To sample from the selected tokens, we must convert their logits into probabilities using Softmax and then use `torch.multinomial()`.

```python
topk_values, topk_indices = torch.topk(logits, k=N, dim=-1)

topk_probs = torch.softmax(topk_values, dim=-1)

sampled_position = torch.multinomial(topk_probs, num_samples=1)

next_token = topk_indices.gather(-1, sampled_position)
```

Here, `sampled_position` identifies a position within the Top-K candidates, while `next_token` returns the corresponding original token index.

**Key takeaway:** Top-K limits the possible next tokens to the \(K\) highest-scoring candidates, while sampling determines which of those candidates is selected.

In [80]:
last_logit = logits[-1, :].unsqueeze(dim=0) # Get the last logit
topk_values, topk_idxs = torch.topk(last_logit, k=3)
print(f'Last logit values: {last_logit}, shape: {last_logit.shape}')
print(f'Select the top 3 bigger values of the logits: {topk_values} and their IDX are: {topk_idxs}')

Last logit values: tensor([[0.3145, 0.9261, 0.8961, 0.3079, 0.0105]]), shape: torch.Size([1, 5])
Select the top 3 bigger values of the logits: tensor([[0.9261, 0.8961, 0.3145]]) and their IDX are: tensor([[1, 2, 0]])


## 4.3 Top-P

to do

---

## 5. Loop 


```python

x = [1, 2, 3]

for _ in range(max_tokens)

    # 1. Get the the values of the input forward the model
    logits = model(x) # Normally ther output gonna be (B, T, D_MODEL)
    
    # 2.  Get the last value of the input forward the model
    last_logit = logits[:, -1, :] # Only the last T (B, last_t, d_model)

    # 3. Strategies here
    'greedy or sampling and get the NEXT_TOKEN'

    # 4. Concat (add the new token into the sequence)
    x = x.append([x, next_token], dim = 1) # (OLD TOKENS + NEW TOKEN)

    # 5. See if the token isnt the special token of stop generating loop
    if new_token = eos:
        break



```